In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path_train = "/content/drive/MyDrive/UNSW Datasets/UNSW_NB15_testing-set.csv"
df_train = pd.read_csv(path_train)

path_test = "/content/drive/MyDrive/UNSW Datasets/UNSW_NB15_training-set.csv"
df_test = pd.read_csv(path_test)

In [ ]:
print(df_train.shape)
print(df_test.shape)

(175341, 45)
(82332, 45)


In [ ]:
df = pd.concat([df_train, df_test], axis=0)
# df = df_test
df = df.sample(frac=1)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df['label'].value_counts()

,count
label,
1,164673
0,93000


In [ ]:
df = df.groupby('label')
df = df.sample(n=df.size().min(), replace=True)
df = df.reset_index(drop=True)

In [ ]:
df['label'].value_counts()

,count
label,
0,93000
1,93000


In [ ]:
# list = [i for i in range(11) if i % 2 == 0]
df_columns = df.columns
df_num_columns = df.describe().columns
df_cat_columns = [col for col in df_columns if col not in df_num_columns]

In [ ]:
df_cat_columns

['proto', 'service', 'state', 'attack_cat']

In [ ]:
# df.proto.value_counts()

In [ ]:
def labelEncoder(dataframe, col_name_list):
    from sklearn import preprocessing
    label_encoder = preprocessing.LabelEncoder()
    encoding_dict = {}

    for col_name in col_name_list:
        dataframe[col_name] = label_encoder.fit_transform(dataframe[col_name])
        # Store the mapping for this column
        encoding_dict[col_name] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    return dataframe, encoding_dict

# Apply the function to your datasets
df, encoding_dict_train = labelEncoder(df, df_cat_columns)
df_test, encoding_dict_test = labelEncoder(df_test, df_cat_columns)

In [ ]:
encoding_dict_train

{'proto': {'3pc': np.int64(0),
  'a/n': np.int64(1),
  'aes-sp3-d': np.int64(2),
  'any': np.int64(3),
  'argus': np.int64(4),
  'aris': np.int64(5),
  'arp': np.int64(6),
  'ax.25': np.int64(7),
  'bbn-rcc': np.int64(8),
  'bna': np.int64(9),
  'br-sat-mon': np.int64(10),
  'cbt': np.int64(11),
  'cftp': np.int64(12),
  'chaos': np.int64(13),
  'compaq-peer': np.int64(14),
  'cphb': np.int64(15),
  'cpnx': np.int64(16),
  'crtp': np.int64(17),
  'crudp': np.int64(18),
  'dcn': np.int64(19),
  'ddp': np.int64(20),
  'ddx': np.int64(21),
  'dgp': np.int64(22),
  'egp': np.int64(23),
  'eigrp': np.int64(24),
  'emcon': np.int64(25),
  'encap': np.int64(26),
  'etherip': np.int64(27),
  'fc': np.int64(28),
  'fire': np.int64(29),
  'ggp': np.int64(30),
  'gmtp': np.int64(31),
  'gre': np.int64(32),
  'hmp': np.int64(33),
  'i-nlsp': np.int64(34),
  'iatp': np.int64(35),
  'ib': np.int64(36),
  'icmp': np.int64(37),
  'idpr': np.int64(38),
  'idpr-cmtp': np.int64(39),
  'idrp': np.int64(40

In [ ]:
def labelEncoder(dataframe, col_name_list):
  from sklearn import preprocessing
  label_encoder = preprocessing.LabelEncoder()

  for col_name in col_name_list:
    dataframe[col_name]= label_encoder.fit_transform(dataframe[col_name])

  return dataframe

df = labelEncoder(df, df_cat_columns)
df_test = labelEncoder(df_test, df_cat_columns)

In [ ]:
X = df.drop(['attack_cat', 'label'], axis = 1)
y = df['label']
y_mul = df['attack_cat']

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, shuffle= True, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42, shuffle= True, stratify=y_test)

In [ ]:
X_train.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)

In [ ]:
y_train

,label
0,0
1,0
2,1
3,0
4,0
...,...
148795,0
148796,0
148797,1
148798,1


# 100% Data (Supervised model)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(random_state=0)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
# y_pred = model.predict(X_val)
# accuracy = accuracy_score(y_val, y_pred)
# print(accuracy)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score

classification_report = classification_report(y_test, y_pred, digits=4)
confusion_matrix = confusion_matrix(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred)

# print("Accuracy:", accuracy)
print("Classification report: \n", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

Classification report: 
               precision    recall  f1-score   support

           0     0.9857    0.9940    0.9898      9300
           1     0.9939    0.9856    0.9897      9300

    accuracy                         0.9898     18600
   macro avg     0.9898    0.9898    0.9898     18600
weighted avg     0.9898    0.9898    0.9898     18600

Confusion matrix: [[9244   56]
 [ 134 9166]]
AUC score: 0.9897849462365592


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score

y_pred = model.predict(X_val)
classification_report = classification_report(y_val, y_pred, digits=4)
confusion_matrix = confusion_matrix(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

# print("Accuracy:", accuracy)
print("Classification report: \n", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

Classification report: 
               precision    recall  f1-score   support

           0     0.9861    0.9918    0.9890      9300
           1     0.9918    0.9860    0.9889      9300

    accuracy                         0.9889     18600
   macro avg     0.9889    0.9889    0.9889     18600
weighted avg     0.9889    0.9889    0.9889     18600

Confusion matrix: [[9224   76]
 [ 130 9170]]
AUC score: 0.9889247311827956


**20% Labeled Data**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split labeled and unlabeled data
# Assuming 20% of X_train is labeled
labeled_percentage = 0.2
num_labeled = int(len(y_train) * labeled_percentage)

# Labeled data
X_labeled = X_train[:num_labeled]
y_labeled = y_train[:num_labeled]

# Unlabeled data
X_unlabeled = X_train[num_labeled:]
y_unlabeled = -1 * np.ones(len(X_unlabeled), dtype=int)  # Assign -1 for unlabeled samples

# Combine labeled and unlabeled data
X_combined = np.vstack((X_labeled, X_unlabeled))
y_combined = np.hstack((y_labeled, y_unlabeled))

# Define the base classifier
base_model = RandomForestClassifier(random_state=42)

# Initialize self-training model
self_training_model = SelfTrainingClassifier(base_model, criterion='k_best', k_best=10)

# Fit the self-training model
self_training_model.fit(X_combined, y_combined)

# Evaluate on validation set
y_val_pred = self_training_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Classification Report:\n", classification_report(y_val, y_val_pred, digits=4))

# Evaluate on test set
y_test_pred = self_training_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Classification Report:\n", classification_report(y_test, y_test_pred, digits=4))


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Validation Accuracy: 0.9712365591397849
Validation Classification Report:
               precision    recall  f1-score   support

           0     0.9674    0.9754    0.9714      9300
           1     0.9752    0.9671    0.9711      9300

    accuracy                         0.9712     18600
   macro avg     0.9713    0.9712    0.9712     18600
weighted avg     0.9713    0.9712    0.9712     18600



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Test Accuracy: 0.9746774193548388
Test Classification Report:
               precision    recall  f1-score   support

           0     0.9697    0.9800    0.9748      9300
           1     0.9798    0.9694    0.9745      9300

    accuracy                         0.9747     18600
   macro avg     0.9747    0.9747    0.9747     18600
weighted avg     0.9747    0.9747    0.9747     18600



**50% Labeled Data**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split labeled and unlabeled data
# Assuming 20% of X_train is labeled
labeled_percentage = 0.5
num_labeled = int(len(y_train) * labeled_percentage)

# Labeled data
X_labeled = X_train[:num_labeled]
y_labeled = y_train[:num_labeled]

# Unlabeled data
X_unlabeled = X_train[num_labeled:]
y_unlabeled = -1 * np.ones(len(X_unlabeled), dtype=int)  # Assign -1 for unlabeled samples

# Combine labeled and unlabeled data
X_combined = np.vstack((X_labeled, X_unlabeled))
y_combined = np.hstack((y_labeled, y_unlabeled))

# Define the base classifier
base_model = RandomForestClassifier(random_state=42)

# Initialize self-training model
self_training_model = SelfTrainingClassifier(base_model, criterion='k_best', k_best=10)

# Fit the self-training model
self_training_model.fit(X_combined, y_combined)

# Evaluate on validation set
y_val_pred = self_training_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Classification Report:\n", classification_report(y_val, y_val_pred, digits=4))

# Evaluate on test set
y_test_pred = self_training_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Classification Report:\n", classification_report(y_test, y_test_pred, digits=4))


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Validation Accuracy: 0.9818817204301076
Validation Classification Report:
               precision    recall  f1-score   support

           0     0.9775    0.9865    0.9820      9300
           1     0.9863    0.9773    0.9818      9300

    accuracy                         0.9819     18600
   macro avg     0.9819    0.9819    0.9819     18600
weighted avg     0.9819    0.9819    0.9819     18600



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Test Accuracy: 0.9844623655913979
Test Classification Report:
               precision    recall  f1-score   support

           0     0.9799    0.9892    0.9845      9300
           1     0.9891    0.9797    0.9844      9300

    accuracy                         0.9845     18600
   macro avg     0.9845    0.9845    0.9845     18600
weighted avg     0.9845    0.9845    0.9845     18600



**5% Labeled Data**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split labeled and unlabeled data
# Assuming 20% of X_train is labeled
labeled_percentage = 0.05
num_labeled = int(len(y_train) * labeled_percentage)

# Labeled data
X_labeled = X_train[:num_labeled]
y_labeled = y_train[:num_labeled]

# Unlabeled data
X_unlabeled = X_train[num_labeled:]
y_unlabeled = -1 * np.ones(len(X_unlabeled), dtype=int)  # Assign -1 for unlabeled samples

# Combine labeled and unlabeled data
X_combined = np.vstack((X_labeled, X_unlabeled))
y_combined = np.hstack((y_labeled, y_unlabeled))

# Define the base classifier
base_model = RandomForestClassifier(random_state=42)

# Initialize self-training model
self_training_model = SelfTrainingClassifier(base_model, criterion='k_best', k_best=10)

# Fit the self-training model
self_training_model.fit(X_combined, y_combined)

# Evaluate on validation set
y_val_pred = self_training_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Classification Report:\n", classification_report(y_val, y_val_pred, digits=4))

# Evaluate on test set
y_test_pred = self_training_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Classification Report:\n", classification_report(y_test, y_test_pred, digits=4))


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Validation Accuracy: 0.9554301075268817
Validation Classification Report:
               precision    recall  f1-score   support

           0     0.9528    0.9583    0.9556      9300
           1     0.9580    0.9526    0.9553      9300

    accuracy                         0.9554     18600
   macro avg     0.9554    0.9554    0.9554     18600
weighted avg     0.9554    0.9554    0.9554     18600



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Test Accuracy: 0.9576344086021505
Test Classification Report:
               precision    recall  f1-score   support

           0     0.9584    0.9568    0.9576      9300
           1     0.9568    0.9585    0.9577      9300

    accuracy                         0.9576     18600
   macro avg     0.9576    0.9576    0.9576     18600
weighted avg     0.9576    0.9576    0.9576     18600



**10% Labeled Data**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split labeled and unlabeled data
# Assuming 20% of X_train is labeled
labeled_percentage = 0.1
num_labeled = int(len(y_train) * labeled_percentage)

# Labeled data
X_labeled = X_train[:num_labeled]
y_labeled = y_train[:num_labeled]

# Unlabeled data
X_unlabeled = X_train[num_labeled:]
y_unlabeled = -1 * np.ones(len(X_unlabeled), dtype=int)  # Assign -1 for unlabeled samples

# Combine labeled and unlabeled data
X_combined = np.vstack((X_labeled, X_unlabeled))
y_combined = np.hstack((y_labeled, y_unlabeled))

# Define the base classifier
base_model = RandomForestClassifier(random_state=42)

# Initialize self-training model
self_training_model = SelfTrainingClassifier(base_model, criterion='k_best', k_best=10)

# Fit the self-training model
self_training_model.fit(X_combined, y_combined)

# Evaluate on validation set
y_val_pred = self_training_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Classification Report:\n", classification_report(y_val, y_val_pred, digits=4))

# Evaluate on test set
y_test_pred = self_training_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Classification Report:\n", classification_report(y_test, y_test_pred, digits=4))


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Validation Accuracy: 0.9636021505376344
Validation Classification Report:
               precision    recall  f1-score   support

           0     0.9604    0.9671    0.9637      9300
           1     0.9669    0.9601    0.9635      9300

    accuracy                         0.9636     18600
   macro avg     0.9636    0.9636    0.9636     18600
weighted avg     0.9636    0.9636    0.9636     18600



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Test Accuracy: 0.9655913978494624
Test Classification Report:
               precision    recall  f1-score   support

           0     0.9627    0.9687    0.9657      9300
           1     0.9685    0.9625    0.9655      9300

    accuracy                         0.9656     18600
   macro avg     0.9656    0.9656    0.9656     18600
weighted avg     0.9656    0.9656    0.9656     18600



**1% Labeled Data**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split labeled and unlabeled data
# Assuming 20% of X_train is labeled
labeled_percentage = 0.01
num_labeled = int(len(y_train) * labeled_percentage)

# Labeled data
X_labeled = X_train[:num_labeled]
y_labeled = y_train[:num_labeled]

# Unlabeled data
X_unlabeled = X_train[num_labeled:]
y_unlabeled = -1 * np.ones(len(X_unlabeled), dtype=int)  # Assign -1 for unlabeled samples

# Combine labeled and unlabeled data
X_combined = np.vstack((X_labeled, X_unlabeled))
y_combined = np.hstack((y_labeled, y_unlabeled))

# Define the base classifier
base_model = RandomForestClassifier(random_state=42)

# Initialize self-training model
self_training_model = SelfTrainingClassifier(base_model, criterion='k_best', k_best=10)

# Fit the self-training model
self_training_model.fit(X_combined, y_combined)

# Evaluate on validation set
y_val_pred = self_training_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Classification Report:\n", classification_report(y_val, y_val_pred, digits=4))

# Evaluate on test set
y_test_pred = self_training_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Classification Report:\n", classification_report(y_test, y_test_pred, digits=4))


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelfTrainingClassifier was fitted without feature names
  warnings.warn(


Validation Accuracy: 0.938494623655914
Validation Classification Report:
               precision    recall  f1-score   support

           0     0.9369    0.9403    0.9386      9300
           1     0.9401    0.9367    0.9384      9300

    accuracy                         0.9385     18600
   macro avg     0.9385    0.9385    0.9385     18600
weighted avg     0.9385    0.9385    0.9385     18600

Test Accuracy: 0.9389784946236559
Test Classification Report:
               precision    recall  f1-score   support

           0     0.9419    0.9357    0.9388      9300
           1     0.9361    0.9423    0.9392      9300

    accuracy                         0.9390     18600
   macro avg     0.9390    0.9390    0.9390     18600
weighted avg     0.9390    0.9390    0.9390     18600

